# Khảo sát chi tiết RAG Core (Phase 3)

Notebook này phân tách các bước của hệ thống Agentic RAG pháp luật giao thông thành các bước nhỏ để dễ phân tích và hiểu rõ cơ chế hoạt động của Hybrid Search và LLM Generator.

## 1. Nạp các thư viện và cấu hình cần thiết

Chạy cell này để khai báo module RAG Core.

In [1]:
# Cấu hình hiển thị để không bị ngắt quãng hoặc giới hạn độ dài
from IPython.display import display, HTML
display(HTML("<style>.jp-OutputArea-child { max-height: unset !important; } .jp-OutputArea-output { max-height: unset !important; }</style>"))
print("[OK] Đã tắt giới hạn chiều cao output (scroll). Hãy chạy tiếp các cell bên dưới.")

[OK] Đã tắt giới hạn chiều cao output (scroll). Hãy chạy tiếp các cell bên dưới.


In [2]:
import sys
import os
from pathlib import Path

# Đảm bảo import được thư mục source
BASE_DIR = Path().resolve().parent if Path().resolve().name in ["notebooks", "tests"] else Path().resolve()
sys.path.insert(0, str(BASE_DIR / "source"))

from rag_core import TrafficHybridRetriever, LegalAnswerGenerator

print("[OK] Thư viện RAG đã sẵn sàng!")

/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[OK] Thư viện RAG đã sẵn sàng!


## 2. Thử nghiệm Truy xuất thông tin (Retrieval)

Ở bước này, chúng ta sẽ xem **TrafficHybridRetriever** hoạt động ra sao. Retriever sử dụng **Hybrid Search** (kết hợp Vector của Qdrant và Keyword của BM25) bằng thuật toán phân giải RRF (Reciprocal Rank Fusion).

*Lưu ý: Qdrant Docker Container cần đang chạy trên cổng 6334.*

In [3]:
# Khởi tạo Retriever
retriever = TrafficHybridRetriever()

# Thử nghiệm 1 câu hỏi đang bị nhiễu (khiến Vector dễ nhầm)
query = "Chu kỳ đăng kiểm lần đầu cho xe ô tô con không kinh doanh vận tải sản xuất năm 2025?"

print(f"\nCÂU HỎI:\n{query}")
print("-" * 80)

# Lấy 5 chunks liên quan nhất
chunks = retriever.get_relevant_chunks(query, top_k=5)

print(f"TÌM THẤY {len(chunks)} ĐOẠN VĂN MẪU:\n")
for i, chunk in enumerate(chunks, 1):
    doc_id = chunk.metadata.get('doc_id', '?')
    dieu = chunk.metadata.get('dieu', '?')
    print(f"[{i}] Điểm RRF: {chunk.score:.4f} | Văn bản: {doc_id} | Điều {dieu}")
    print(f"    Preview: {chunk.content[:150]}...\n")

/media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/traffic_rag/source/rag_core/retriever.py:80: UserWarning: Qdrant client version 1.14.2 is incompatible with server version 1.17.0. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  self.client = QdrantClient(host=qdrant_host, port=qdrant_port)



CÂU HỎI:
Chu kỳ đăng kiểm lần đầu cho xe ô tô con không kinh doanh vận tải sản xuất năm 2025?
--------------------------------------------------------------------------------
TÌM THẤY 41 ĐOẠN VĂN MẪU:

[1] Điểm RRF: 0.0303 | Văn bản: 47/2024/TT-BGTVT | Điều 35
    Preview: Văn bản: Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) | Điều 35: Hiệu lực và trách nhiệm thi hành
1. Nguyên tắc xác định chu kỳ kiểm địn...

[2] Điểm RRF: 0.0300 | Văn bản: 47/2024/TT-BGTVT | Điều 4
    Preview: Văn bản: Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) | Điều 4: Địa điểm thực hiện kiểm định, đối tượng miễn kiểm định lần đầu
3. Đối tư...

[3] Điểm RRF: 0.0284 | Văn bản: 79/2024/TT-BCA | Điều 36
    Preview: Văn bản: Thông tư 79/2024/TT-BCA (Đăng ký xe) | Điều 36: Xác định năm sản xuất của xe
## Điều 36. Xác định năm sản xuất của xe
1. Năm sản xuất của xe ...

[4] Điểm RRF: 0.0276 | Văn bản: 79/2024/TT-BCA | Điều 14
    Preview: Văn bản: Thông tư 79/2024/TT-BCA (Đăng ký xe) | Đ

## 3. Khởi tạo LLM Generator (LangChain)

Tiếp theo, chúng ta cắm một LLM vào thay cho con người để hệ thống tự đọc văn bản được truy xuất ở trên và sinh ra câu trả lời.

Bạn cần thiết lập khóa API cho OpenAI hoặc Google :

In [4]:
import getpass
from pathlib import Path


def _load_dotenv_simple() -> Path | None:
    """Tìm và nạp file .env từ thư mục hiện tại hoặc bất kỳ thư mục cha nào."""
    for parent in [Path().resolve(), *Path().resolve().parents]:
        candidate = parent / ".env"
        if candidate.exists():
            for line in candidate.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                k, v = line.split("=", 1)
                k = k.strip()
                v = v.strip().strip('"').strip("'")
                # Ưu tiên .env → ghi đè biến cũ trong session có thể đã stale/sai
                os.environ[k] = v
            return candidate
    return None


env_path = _load_dotenv_simple()
if env_path:
    print(f"[OK] Đã nạp biến môi trường từ {env_path}")

# File .env dự án dùng tên "API_KEY" — map sang các tên SDK thực tế.
_shared_key = os.environ.get("API_KEY")
if _shared_key:
    os.environ.setdefault("GOOGLE_API_KEY", _shared_key)
    os.environ.setdefault("GEMINI_API_KEY", _shared_key)

provider = "google"  # Chuyển đổi giữa "openai" hoặc "google"

# Model ID chính xác trên Gemini API (xác minh qua ListModels):
#   - Dashboard: "Gemini 3.1 Flash Lite"  →  API id: "gemini-3.1-flash-lite-preview"
# Quota free tier: 15 RPM / 250K TPM / 500 RPD.
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"

if provider == "openai":
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Nhập OPENAI_API_KEY: ")
    generator = LegalAnswerGenerator(provider="openai", model="gpt-4o-mini")
    print("Đã khởi tạo OpenAI (GPT-4o-Mini)!")
else:
    if not os.environ.get("GOOGLE_API_KEY"):
        os.environ["GOOGLE_API_KEY"] = getpass.getpass("Nhập GOOGLE_API_KEY: ")
    google_key = os.environ["GOOGLE_API_KEY"]
    print(f"[debug] GOOGLE_API_KEY (masked): {google_key[:6]}...{google_key[-4:]}")
    generator = LegalAnswerGenerator(provider="google", model=GEMINI_MODEL)
    print(f"Đã khởi tạo Google Gemini ({GEMINI_MODEL})!")

[OK] Đã nạp biến môi trường từ /media/pphong/D:/Do_An_Tot_Nghiep/GitHub1/.env
[debug] GOOGLE_API_KEY (masked): AIzaSy...Vz6s


/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.12) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/pphong/venv/LLM_Agentic/lib/python3.10/site-packages/langchain_google_genai/chat_models.py:47: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  from google.generativeai.caching import CachedContent  # type: ignore[import]


Đã khởi tạo Google Gemini (gemini-3.1-flash-lite-preview)!


## 4. Xây dựng câu trả lời (LLM Generation)

Chúng ta truyền trực tiếp kết quả `chunks` lấy được từ `retriever` vào `generator`. Model sẽ đọc các chunk đó, định dạng lại thành số liệu và trích dẫn chuẩn pháp luật Việt Nam.

In [5]:
print(f"Gửi ngữ cảnh đến {generator.model_name}...")

# Chuyển đổi đối tượng data format
chunk_dicts = [c.to_dict() for c in chunks]

# Yêu cầu LLM trả lời
out = generator.generate(query, chunk_dicts)

print("\n========================== ĐÁP ÁN TỪ AI ==========================\n")
print(out['answer'])
print("\n========================== SIÊU DỮ LIỆU CẮM UI ==================\n")
print("Liệt kê các dòng trích dẫn gốc:\n")
for s in out['sources']:
    print(f" - Điều {s.get('dieu', '?')} - {s.get('ten_van_ban', '')} ({s.get('doc_id')})")

if out['refused']:
    print("\n[Cảnh báo]: LLM đã từ chối trả lời do thiếu thông tin pháp lý trong Context!")

Gửi ngữ cảnh đến gemini-3.1-flash-lite-preview...

========================== ĐÁP ÁN TỪ AI ==========================

Chu kỳ kiểm định lần đầu cho xe ô tô chở người đến 08 chỗ (không kể chỗ của người lái xe) không kinh doanh vận tải là 36 tháng [Điều 35, Khoản 2 — Thông tư 47/2024/TT-BGTVT (47/2024/TT-BGTVT)].

========================== SIÊU DỮ LIỆU CẮM UI ==================

Liệt kê các dòng trích dẫn gốc:

 - Điều 35 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 4 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 35 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 35 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 35 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 35 - Thông tư 47/2024/TT-BGTVT (Trạm kiểm định khí thải xe máy) (47/2024/TT-BGTVT)
 - Điều 35 - Thông tư 47/2024/TT-BGTVT (T

## 5. Batch Evaluation (Test Suite)

Mục này chạy một loạt truy vấn khó trên pipeline đã khởi tạo ở các section trên (`retriever` + `generator`) để đánh giá độ chính xác trên nhiều nhóm tài liệu:

- **Xử phạt** (NĐ 168/2024): mức phạt vượt đèn đỏ, nồng độ cồn, số điểm GPLX bị trừ.
- **Kỹ thuật** (TT 47/2024 và các TT về cải tạo xe): chu kỳ đăng kiểm, điều kiện cải tạo.
- **Thủ tục hành chính** (TT 79/2024): đăng ký xe online.
- **Kịch bản đa văn bản**: LLM phải tổng hợp Luật 36/2024/QH15 và NĐ 168/2024 trong cùng một câu trả lời.
- **Ngoài phạm vi** (Bộ luật Hình sự): kỳ vọng `refused=True` và câu từ chối _"Thông tin này không có trong tài liệu được cung cấp."_

Điều kiện tiên quyết: các cell Section 1–4 phải đã chạy thành công để có sẵn biến `retriever` và `generator`.


In [6]:
import time
import traceback

# Ghi chú retrieval: corpus KHÔNG chứa cụm khẩu ngữ "vượt đèn đỏ" (0 hit BM25);
# thuật ngữ pháp lý dùng trong NĐ 168/2024 là "không chấp hành hiệu lệnh của đèn
# tín hiệu giao thông". Query bổ sung keyword pháp lý + doc_id để retriever match.
TOP_K = 8

TEST_QUERIES = [
    {
        "category": "Xử phạt - Vượt đèn đỏ",
        "query": (
            "Vượt đèn đỏ (không chấp hành hiệu lệnh của đèn tín hiệu giao thông) "
            "khi điều khiển ô tô bị phạt bao nhiêu tiền và trừ bao nhiêu điểm GPLX "
            "theo Nghị định 168/2024/NĐ-CP?"
        ),
    },
    {
        "category": "Xử phạt - Nồng độ cồn",
        "query": "Mức phạt đối với người điều khiển xe mô tô, xe máy có nồng độ cồn vượt quá 0,4 miligam/1 lít khí thở là bao nhiêu tiền và bị trừ bao nhiêu điểm giấy phép lái xe theo Nghị định 168/2024/NĐ-CP?",
    },
    {
        "category": "Kỹ thuật - Chu kỳ đăng kiểm",
        "query": "Chu kỳ kiểm định định kỳ của xe ô tô chở người đến 9 chỗ kinh doanh vận tải đã sản xuất trên 5 năm là bao lâu?",
    },
    {
        "category": "Kỹ thuật - Cải tạo xe",
        "query": "Điều kiện và hồ sơ để cải tạo xe cơ giới gồm những gì?",
    },
    {
        "category": "Thủ tục - Đăng ký xe online",
        "query": "Hồ sơ đăng ký xe lần đầu qua dịch vụ công trực tuyến cần những giấy tờ gì theo Thông tư 79/2024/TT-BCA?",
    },
    {
        "category": "Tổng hợp đa văn bản",
        "query": "Khi Luật Trật tự, an toàn giao thông đường bộ 2024 có hiệu lực, hành vi không chấp hành hiệu lệnh đèn tín hiệu bị xử lý thế nào về hình thức phạt tiền và trừ điểm giấy phép lái xe theo Nghị định 168/2024/NĐ-CP?",
    },
    {
        "category": "Ngoài phạm vi (kỳ vọng từ chối)",
        "query": "Tội giết người theo Bộ luật Hình sự bị xử phạt như thế nào?",
    },
]

# gemini-3.1-flash-lite-preview free tier = 15 RPM → sleep 5s giữa các query là đủ an toàn.
SLEEP_BETWEEN_QUERIES = 5


def _fmt_chunk_loc(meta: dict) -> str:
    parts = [meta.get("doc_id", "?"), f"Điều {meta.get('dieu', '?')}"]
    if meta.get("khoan") is not None:
        parts.append(f"K{meta['khoan']}")
    if meta.get("diem") is not None:
        parts.append(f"Đ{meta['diem']}")
    return "/".join(parts)


for i, case in enumerate(TEST_QUERIES, 1):
    print("=" * 90)
    print(f"[TEST {i}/{len(TEST_QUERIES)}] {case['category']}")
    print(f"CÂU HỎI: {case['query']}")
    print("-" * 90)

    try:
        chunks = retriever.get_relevant_chunks(case["query"], top_k=TOP_K)
        print(f"[retrieved {len(chunks)} chunks]")
        for j, c in enumerate(chunks, 1):
            print(f"  #{j} score={c.score:.4f} | {_fmt_chunk_loc(c.metadata)}")

        result = generator.generate(case["query"], [c.to_dict() for c in chunks])

        print("\nTRẢ LỜI:")
        print(result["answer"])

        print("\nTRÍCH DẪN:")
        if result["sources"]:
            for s in result["sources"]:
                parts = [f"{s.get('doc_id', '?')}", f"Điều {s.get('dieu', '?')}"]
                if s.get("khoan") is not None:
                    parts.append(f"Khoản {s['khoan']}")
                if s.get("diem") is not None:
                    parts.append(f"Điểm {s['diem']}")
                ten = s.get("ten_van_ban", "")
                print(f"  - {' | '.join(parts)} — {ten}")
        else:
            print("  (không có trích dẫn)")

        print(f"REFUSED FLAG: {result['refused']}")
    except Exception as e:
        print(f"[ERROR] {type(e).__name__}: {e}")
        traceback.print_exc(limit=2)

    print()
    if i < len(TEST_QUERIES):
        time.sleep(SLEEP_BETWEEN_QUERIES)


[TEST 1/7] Xử phạt - Vượt đèn đỏ
CÂU HỎI: Vượt đèn đỏ (không chấp hành hiệu lệnh của đèn tín hiệu giao thông) khi điều khiển ô tô bị phạt bao nhiêu tiền và trừ bao nhiêu điểm GPLX theo Nghị định 168/2024/NĐ-CP?
------------------------------------------------------------------------------------------
[retrieved 65 chunks]
  #1 score=0.0295 | 168/2024/NĐ-CP/Điều 6/K9
  #2 score=0.0287 | 168/2024/NĐ-CP/Điều 7/K1/Đb
  #3 score=0.0286 | 168/2024/NĐ-CP/Điều 8/K7
  #4 score=0.0282 | 168/2024/NĐ-CP/Điều 9/K1/Đc
  #5 score=0.0266 | 168/2024/NĐ-CP/Điều 6/K5/Đa
  #6 score=0.0161 | 168/2024/NĐ-CP/Điều 7/K1/Đe
  #7 score=0.0159 | 168/2024/NĐ-CP/Điều 9/K1/Đd
  #8 score=0.0159 | 168/2024/NĐ-CP/Điều 10
  #9 score=0.0000 | 168/2024/NĐ-CP/Điều 6/K14
  #10 score=0.0000 | 168/2024/NĐ-CP/Điều 6/K15
  #11 score=0.0000 | 168/2024/NĐ-CP/Điều 6/K16
  #12 score=0.0000 | 168/2024/NĐ-CP/Điều 7/K11
  #13 score=0.0000 | 168/2024/NĐ-CP/Điều 7/K12
  #14 score=0.0000 | 168/2024/NĐ-CP/Điều 7/K13
  #15 score=0.0000 | 1